# Benchmark del minigestor

Borrador sin ejecutar. Usa los CSV de `data/generated/`, creados con `python data/generate_data.py`. Las conexiones pendientes están en `None`: no producen tiempos ni resultados simulados.

Para una comparación final habrá que fijar la misma carga, tamaños de página y condiciones de ejecución para cada estructura.

In [ ]:
import csv
import statistics
import sys
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
from time import perf_counter

ROOT = Path.cwd()
if ROOT.name == 'benchmarks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'generated'
OUTPUT = ROOT / 'data' / 'benchmarks'
sys.path.insert(0, str(ROOT / 'backend'))
from engine.hashing import ExtendibleHashIndex, external_group_by, grace_hash_join

SIZES = (1_000, 10_000, 100_000)
REPEATS = 3
RUN_BENCHMARKS = False  # cambiar a True cuando se decida ejecutar
results = []

## Carga y comprobación de datos

La carga del CSV y la validación ocurren fuera del tiempo medido. Cada `record_id` de `details` debe existir en `records`.

In [ ]:
def load_data(size):
    with (DATA / f'records_{size}.csv').open(newline='') as file:
        records = [tuple(map(int, row)) for row in list(csv.reader(file))[1:]]
    with (DATA / f'details_{size}.csv').open(newline='') as file:
        details = [tuple(map(int, row)) for row in list(csv.reader(file))[1:]]
    ids = {row[0] for row in records}
    assert len(records) == len(ids) == size
    assert len(details) == 2 * size
    assert all(row[1] in ids for row in details)
    return records, details

def add_result(suite, structure, operation, size, run, seconds, disk_bytes=None):
    results.append((suite, structure, operation, size, run, seconds, disk_bytes))

## Conexiones pendientes

Sustituir cada `None` por una función que reciba `(records, details, run)` y agregue mediciones con `add_result`. Esas funciones deben validar sus resultados antes de registrarlos.

| Estructura | Operaciones previstas |
|---|---|
| Heap / secuencial | inserción, búsqueda, eliminación, espacio y reorganización |
| B+ agrupado / no agrupado | construcción, igualdad, rango, ordenamiento y eliminación |
| External sort | `ORDER BY` y espacio temporal |

El heap actual requiere corrección antes de medirlo; el secuencial, los B+ y external sort aún no tienen las operaciones necesarias.

In [ ]:
PENDING = {
    'heap': None,
    'sequential': None,
    'bplus_clustered': None,
    'bplus_unclustered': None,
    'external_sort': None,
}

## Índice hash extensible

Mide construcción, 100 búsquedas exactas y 100 eliminaciones. Los archivos del índice se crean en un directorio temporal por repetición.

In [ ]:
def benchmark_hash(records, size, run):
    keys = list(range(0, size, max(1, size // 100)))[:100]
    with tempfile.TemporaryDirectory() as tmp:
        path = str(Path(tmp) / 'index')
        start = perf_counter()
        index = ExtendibleHashIndex(path, key_type='int', block_factor=64)
        try:
            for key, *_ in records:
                index.insert(key, (key, 0))
            build_seconds = perf_counter() - start
            assert index.stats()['entries'] == size
            assert all(index.search(key) == [(key, 0)] for key in keys)
            disk_bytes = sum(path.stat().st_size for path in Path(tmp).iterdir())
            add_result('indexes', 'extendible_hash', 'build', size, run, build_seconds, disk_bytes)

            start = perf_counter()
            found = [index.search(key) for key in keys]
            search_seconds = perf_counter() - start
            assert found == [[(key, 0)] for key in keys]
            add_result('indexes', 'extendible_hash', 'equality_100', size, run, search_seconds, disk_bytes)

            start = perf_counter()
            removed = [index.delete(key) for key in keys]
            delete_seconds = perf_counter() - start
            assert all(removed) and all(not index.search(key) for key in keys)
            add_result('indexes', 'extendible_hash', 'delete_100', size, run, delete_seconds, disk_bytes)
        finally:
            index.close()

## Algoritmos externos disponibles

`GROUP BY` agrupa por `category`. `JOIN` une `records.id = details.record_id`. Se consume el resultado completo dentro del tiempo medido.

In [ ]:
def benchmark_external(records, details, size, run):
    expected_groups = Counter(row[1] for row in records)
    with tempfile.TemporaryDirectory() as tmp:
        start = perf_counter()
        groups = dict(external_group_by(records, lambda row: row[1], [('count', None)], mem_budget=20, tmp_dir=tmp))
        group_seconds = perf_counter() - start
        assert {key: value[0] for key, value in groups.items()} == dict(expected_groups)
        add_result('external', 'external_hash', 'group_by', size, run, group_seconds)

    with tempfile.TemporaryDirectory() as tmp:
        start = perf_counter()
        joined = list(grace_hash_join(records, details, lambda row: row[0], lambda row: row[1], mem_budget=1_000, tmp_dir=tmp))
        join_seconds = perf_counter() - start
        assert len(joined) == 2 * size
        assert all(parent[0] == child[1] for parent, child in joined)
        add_result('external', 'external_hash', 'join', size, run, join_seconds)

## Ejecución

Se ejecuta solo cuando `RUN_BENCHMARKS = True`. Cada repetición usa archivos nuevos; los CSV de entrada no se generan ni modifican aquí.

In [ ]:
if RUN_BENCHMARKS:
    results.clear()
    for size in SIZES:
        records, details = load_data(size)
        for run in range(1, REPEATS + 1):
            benchmark_hash(records, size, run)
            benchmark_external(records, details, size, run)
            for name, benchmark in PENDING.items():
                if benchmark is not None:
                    benchmark(records, details, run)
    print(f'{len(results)} mediciones; pendientes: {[name for name, fn in PENDING.items() if fn is None]}')

## CSV y gráficas

Se guardan resultados crudos y medianas. Las gráficas muestran únicamente las operaciones medidas. Instalar `matplotlib` cuando se ejecuten las gráficas.

In [ ]:
if results:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    with (OUTPUT / 'results_raw.csv').open('w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(('suite', 'structure', 'operation', 'records', 'run', 'seconds', 'disk_bytes'))
        writer.writerows(results)

    samples = defaultdict(list)
    for suite, structure, operation, size, run, seconds, disk_bytes in results:
        samples[(suite, structure, operation, size)].append((seconds, disk_bytes))
    summary = [(suite, structure, operation, size, statistics.median(t for t, _ in values),
                statistics.median(b for _, b in values if b is not None) if any(b is not None for _, b in values) else '')
               for (suite, structure, operation, size), values in sorted(samples.items())]
    with (OUTPUT / 'results_summary.csv').open('w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(('suite', 'structure', 'operation', 'records', 'median_seconds', 'median_disk_bytes'))
        writer.writerows(summary)

    import matplotlib.pyplot as plt
    for operation in sorted({row[2] for row in summary}):
        fig, ax = plt.subplots()
        for structure in sorted({row[1] for row in summary if row[2] == operation}):
            points = [(row[3], row[4]) for row in summary if row[1] == structure and row[2] == operation]
            ax.plot(*zip(*points), marker='o', label=structure)
        ax.set(xlabel='Registros', ylabel='Mediana (s)', title=operation)
        ax.legend()
        fig.savefig(OUTPUT / f'{operation}.png', bbox_inches='tight')
        plt.close(fig)
    disk_rows = [row for row in summary if row[5] != '']
    if disk_rows:
        fig, ax = plt.subplots()
        for structure in sorted({row[1] for row in disk_rows}):
            points = sorted((row[3], row[5]) for row in disk_rows if row[1] == structure and row[2] == 'build')
            if points:
                ax.plot(*zip(*points), marker='o', label=structure)
        ax.set(xlabel='Registros', ylabel='Bytes', title='Espacio de índices')
        ax.legend()
        fig.savefig(OUTPUT / 'index_disk.png', bbox_inches='tight')
        plt.close(fig)